# TAELTX-2 Encoding and Decoding Demo

This notebook tests encoding & decoding compatibility of TAELTX-2 with the LTX-2 VAE.

## Video Downloading

In [1]:
!python3 -m pip install --quiet yt-dlp

In [2]:
!yt-dlp --no-warnings 'https://www.youtube.com/shorts/LLXK7ub_QWQ' -o big_cake

[youtube] Extracting URL: https://www.youtube.com/shorts/LLXK7ub_QWQ
[youtube] LLXK7ub_QWQ: Downloading webpage
[youtube] LLXK7ub_QWQ: Downloading android sdkless player API JSON
[youtube] LLXK7ub_QWQ: Downloading web safari player API JSON
[youtube] LLXK7ub_QWQ: Downloading m3u8 information
[info] LLXK7ub_QWQ: Downloading 1 format(s): 399+251
[download] big_cake.webm has already been downloaded


In [3]:
!ffmpeg -y -i "big_cake."* -t 2 -v quiet -hide_banner -stats \
  -vf "crop=min(iw\,ih):min(iw\,ih),scale=256:256" \
  -c:v libx264 -b:v 5M -pix_fmt yuv420p -an -r 30 \
  small_cake.mp4

frame=   60 fps=0.0 q=-1.0 Lsize=     720kB time=00:00:01.90 bitrate=3104.2kbits/s dup=0 drop=57 speed=9.88x    


In [4]:
!du -sh small_cake.mp4

720K	small_cake.mp4


In [5]:
from IPython.display import Video, Markdown
import torch as th
import torch.nn as nn

Video("small_cake.mp4", html_attributes="playsinline autoplay loop muted", embed=True)

# Make VAE Testing Code

In [6]:
import cv2

class VideoTensorReader:
    def __init__(self, video_file_path):
        self.cap = cv2.VideoCapture(video_file_path)
        assert self.cap.isOpened(), f"Could not load {video_file_path}"
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
    def __iter__(self):
        return self
    def __next__(self):
        ret, frame = self.cap.read()
        if not ret:
            self.cap.release()
            raise StopIteration  # End of video or error
        return th.from_numpy(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).permute(2, 0, 1) # BGR HWC -> RGB CHW

class VideoTensorWriter:
    def __init__(self, video_file_path, width_height, fps=30):
        self.writer = cv2.VideoWriter(video_file_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, width_height)
        assert self.writer.isOpened(), f"Could not create writer for {video_file_path}"
    def write(self, frame_tensor):
        assert frame_tensor.ndim == 3 and frame_tensor.shape[0] == 3, f"{frame_tensor.shape}??"
        self.writer.write(cv2.cvtColor(frame_tensor.permute(1, 2, 0).numpy(), cv2.COLOR_RGB2BGR)) # RGB CHW -> BGR HWC
    def __del__(self):
        if hasattr(self, 'writer'): self.writer.release()

@th.no_grad()
def test_vae(vae, video_path="small_cake.mp4", output_path="small_cake_reconstructed.mp4", encoder_vae=None):
    video = th.stack(list(VideoTensorReader(video_path)), 0)[None].to("cuda", th.float16).div_(255.0) # NTCHW [0, 1]
    video = video[:, :8*((video.shape[1]-1)//8)+1] # trim to the magical frame counts that reference VAE doesn't crash on
    # encode
    latent = vae.encode_video(video) if encoder_vae is None else encoder_vae.encode_video(video) # allow separate encoder
    # decode
    output = vae.decode_video(latent)
    # display
    writer = VideoTensorWriter(output_path, output.shape[-2:][::-1])
    for frame in output.mul(255.0).round_().byte().cpu().squeeze(0):
        writer.write(frame)
    del writer
    !ffmpeg -y -i {output_path} -v quiet -hide_banner -stats -c:v libx264 -b:v 5M -pix_fmt yuv420p -an {output_path}.compressed.mp4
    display(Markdown(f"### {output_path}"))
    display(Video(output_path + ".compressed.mp4", html_attributes="playsinline autoplay loop muted controls", embed=True))

# Load Tiny AE, Test

In [7]:
from taehv import TAEHV
tiny_ae = TAEHV("taeltx_2.pth").to("cuda", th.float16).eval().requires_grad_(False)
test_vae(tiny_ae, output_path="tiny_reconstructed.mp4")

  0%|          | 0/18 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

frame=   57 fps=0.0 q=-1.0 Lsize=     700kB time=00:00:01.80 bitrate=3185.7kbits/s speed=13.2x    


### tiny_reconstructed.mp4

# Load Official VAE, Test

In [8]:
class LTX2VAEWrapper(nn.Module):
    def __init__(self):
        super().__init__()
        from diffusers import AutoencoderKLLTX2Video
        self.dtype = th.bfloat16
        self.vae = AutoencoderKLLTX2Video.from_pretrained("Lightricks/LTX-2", subfolder="vae", torch_dtype=self.dtype)

    @th.no_grad()
    def encode_video(self, x):
        assert x.ndim == 5 # NTCHW, [0, 1]
        assert x.shape[2] % 3 == 0 # image channels
        N, T, C, H, W = x.shape
        y = x.transpose(1, 2).to(self.dtype).mul(2).sub_(1) # -> NCTHW, [-1, 1]
        y = self.vae.encode(y).latent_dist.sample()
        return y.sub_(self.vae.latents_mean.view(1, -1, 1, 1, 1)).div_(self.vae.latents_std.view(1, -1, 1, 1, 1)).transpose(1, 2).to(x.dtype) # back to NTCHW
    
    @th.no_grad()
    def decode_video(self, x):
        assert x.ndim == 5 # NTCHW
        assert x.shape[2] % self.vae.config.latent_channels == 0 # latent channels
        N, T, C, H, W = x.shape
        y = x.transpose(1, 2).to(self.dtype).mul_(self.vae.latents_std.view(1, -1, 1, 1, 1)).add_(self.vae.latents_mean.view(1, -1, 1, 1, 1)) # -> NCTHW
        y = self.vae.decode(y).sample
        return y.mul_(0.5).add_(0.5).clamp_(0, 1).transpose(1, 2) # NTCHW [0, 1]

reference_vae = LTX2VAEWrapper().eval().cuda().requires_grad_(False)

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
/home/ubuntu/.local/lib/python3.10/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


In [9]:
test_vae(reference_vae, output_path="reference_reconstructed.mp4")

frame=   57 fps=0.0 q=-1.0 Lsize=     677kB time=00:00:01.80 bitrate=3079.3kbits/s speed=13.9x    


### reference_reconstructed.mp4

# Test tiny AE again, but using the full-scale encoder

In [10]:
test_vae(tiny_ae, encoder_vae=reference_vae, output_path="tiny_decoder_only_reconstructed.mp4")

  0%|          | 0/23 [00:00<?, ?it/s]

frame=   57 fps=0.0 q=-1.0 Lsize=     819kB time=00:00:01.80 bitrate=3727.7kbits/s speed=13.4x    


### tiny_decoder_only_reconstructed.mp4